# Explore possible required agents

For a consultant without a customer agent list, this notebook ranks discovered Software Instance types by semantic similarity to agent concepts and by coverage across Windows or Linux hosts. Results are candidates for customer review, not compliance findings.

The query covers modeled Software Instances, not every installed package. The embedding model runs locally after its first download. Host names and software names are not sent to an LLM service.

## Setup

Install the optional dependency in this notebook's environment with `pip install sentence-transformers`. The first run downloads the model from Hugging Face; use a pre-cached model in restricted environments.

### Suggested embedding models

Set `MODEL_NAME` to one of these identifiers. All three load with the `SentenceTransformer` call below; compare their rankings on a few known agent names before relying on a new model.

| Model | When to try it | Description |
| --- | --- | --- |
| [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | Default | Compact English sentence embedding model; a practical first pass over software names. |
| [`sentence-transformers/all-mpnet-base-v2`](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | Second English ranking | Larger 768-dimensional English embedding model; useful for comparing candidate order when the default misses known agents. Expect more memory and compute. |
| [`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) | Multilingual inventory | Sentence embedding model covering 50 languages; try it when software display names or descriptions are not consistently English. Translate or add agent concept phrases in those languages as needed. |

These models rank text similarity; none establishes that a product is a compliance agent. Model scores are not directly comparable across models.

In [ ]:
import pandas as pd
from tideway import notebooks as tw_nb
from tideway.agent_inventory import HOST_SOFTWARE_QUERY, normalise_hosts, software_coverage

APPLIANCE_NAME = None
APPLIANCE_INDEX = 0
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # See suggested models above
MIN_HOSTS = 20
MIN_COVERAGE = 0.90
TOP_N = 50

AGENT_CONCEPTS = [
    'endpoint protection and antivirus security agent',
    'endpoint detection and response sensor',
    'server monitoring and observability agent',
    'configuration management client agent',
    'vulnerability scanning and asset inventory agent',
    'log collection and forwarding agent',
    'backup and recovery agent',
]

tw = tw_nb.appliance_from_config(appliance_name=APPLIANCE_NAME, appliance_index=APPLIANCE_INDEX)
print('Target:', tw.target)


## Discover software types and prevalence

The same host inventory query powers the customer compliance notebook. Coverage is calculated independently for Windows and Linux.

In [ ]:
response = tw.data().search({'query': HOST_SOFTWARE_QUERY}, format='object', limit=500)
if hasattr(response, 'ok'):
    raise RuntimeError(f'Discovery search failed: {response.status_code} {response.text}')
hosts = normalise_hosts(response)
coverage = pd.DataFrame(software_coverage(hosts))
if coverage.empty:
    raise ValueError('No software types were returned; check Discovery data and the host software query.')
print(f'{len(hosts)} hosts; {coverage["Software Type"].nunique()} distinct software types')
display(coverage.sort_values('Coverage', ascending=False).head(20))


## Semantic candidate ranking

Compare each unique software type with agent concepts. Similarity helps find unfamiliar names; high deployment is supporting evidence. Review results with the customer before using any candidate as a requirement.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)
names = sorted(coverage['Software Type'].unique())
name_vectors = model.encode(names, normalize_embeddings=True, convert_to_numpy=True)
concept_vectors = model.encode(AGENT_CONCEPTS, normalize_embeddings=True, convert_to_numpy=True)
similarities = name_vectors @ concept_vectors.T
best_concepts = similarities.argmax(axis=1)
semantic = pd.DataFrame({
    'Software Type': names,
    'Agent Similarity': similarities.max(axis=1),
    'Closest Concept': [AGENT_CONCEPTS[index] for index in best_concepts],
})
candidates = coverage.merge(semantic, on='Software Type')
candidates = candidates[
    (candidates['Hosts Total'] >= MIN_HOSTS) & (candidates['Coverage'] >= MIN_COVERAGE)
].sort_values(['OS Family', 'Agent Similarity', 'Coverage'], ascending=[True, False, False])
for family in ('Windows', 'Linux'):
    print(f'{family} candidates:')
    display(candidates[candidates['OS Family'] == family].head(TOP_N))


## Review

Similarity is a ranking score, not a probability or proof that software is an agent. High-coverage platform software may appear here. Confirm approved Discovery type names with the customer, then copy them into `REQUIRED_SOFTWARE` in `expected_agents.ipynb`.